In [ ]:
import json
import sys
from typing import List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from simulate_markov_utils import plot_markov_df
from sklearn.metrics import auc, roc_curve

from plotting_notebooks.plotting_utils import (
    load_in_combined_results,
    load_monitoring_data,
    plot_markov_analysis,
    plot_roc_curves,
    plot_suspiciousness_distributions,
)

sys.path.append('..')
from utils.analysis_utils import analyze_all_tasks, extract_verdict
from utils.path_utils import resolve_path

# Constants
MAIN_AGENT_MODELS = ['gpt-4o', 'claude-3-5-sonnet-20241022']
MONITOR_AGENT_MODELS = ['gpt-4o', 'claude-3-5-sonnet-20241022', 'o3-mini', 'o1', 'claude-3-5-haiku-20241022']
DEFAULT_TRANSCRIPT_TYPES = ["transcripts", "transcripts_no_hint", "transcripts_no_secondary_task"]
DOWNSAMPLE_FACTORS = [1, 2, 3, 4, 5, 10]

In [ ]:
def prepare_reflect_on_decision_data(base_dir: str, include_failed_attempts: bool, include_failed_main_task: bool) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load and prepare the initial datasets."""
    # Load monitoring data
    reflect_on_decision_df = load_monitoring_data(base_dir=base_dir)
    for i in range(1, 4):
        reflect_on_decision_df[f'response_{i}_reflection'] = reflect_on_decision_df.data.apply(
            lambda x: extract_verdict(x[f'response_{i}'], tag_word='verdict')
        )
    reflect_on_decision_df['average_suspiciousness_reflection'] = (reflect_on_decision_df['response_1_reflection'] + reflect_on_decision_df['response_2_reflection'] + reflect_on_decision_df['response_3_reflection']) / 3
    reflect_on_decision_df['max_suspiciousness_reflection'] = np.max([reflect_on_decision_df['response_1_reflection'], reflect_on_decision_df['response_2_reflection'], reflect_on_decision_df['response_3_reflection']], axis=0)
    return reflect_on_decision_df

def merge_and_compute_features_for_reflect_on_decision(combined_results: pd.DataFrame, 
                             reflect_on_decision_df: pd.DataFrame) -> pd.DataFrame:
    """Merge the dataframes and compute additional features."""
    merged_df = pd.merge(
        reflect_on_decision_df,
        combined_results,
        on=['task_name', 'transcript_type', 'main_agent', 'monitor_agent', 'file_name'],
        how='left'
    ).dropna()
    
    # Compute features

    merged_df['convo_length'] = merged_df.transcript.apply(lambda x: len(x['conversation']))
    
    return merged_df


In [ ]:

include_failed_attempts = False
include_failed_main_task = False
combined_results = load_in_combined_results(include_failed_attempts, include_failed_main_task)
reflect_on_decision_df = prepare_reflect_on_decision_data(base_dir='monitor_responses/reflect_on_decision_2', include_failed_attempts=include_failed_attempts, include_failed_main_task=include_failed_main_task)
training_data_path = resolve_path('finetuning/training_data_dfs/combined_3.csv')
training_data = pd.read_csv(training_data_path)
reflect_on_decision_df = reflect_on_decision_df[~reflect_on_decision_df.task_name.isin(training_data.task_name)]
reflect_on_decision_df['side_task_evaluation'] = reflect_on_decision_df.transcript_type != 'transcripts_no_secondary_task'

In [ ]:
reflect_on_decision_df['average_suspiciousness'] = (reflect_on_decision_df['response_1_reflection'] + reflect_on_decision_df['response_2_reflection'] + reflect_on_decision_df['response_3_reflection']) / 3
merged_df = merge_and_compute_features_for_reflect_on_decision(combined_results, reflect_on_decision_df)
plot_roc_curves(reflect_on_decision_df, combined_results, give_hint=None, include_failed_attempts=include_failed_attempts, include_failed_main_task=include_failed_main_task, monitors = reflect_on_decision_df.monitor_agent.unique(), metrics = ['response_1_reflection', 'response_2_reflection', 'response_3_reflection', 'average_suspiciousness_reflection', 'max_suspiciousness_reflection'])
plot_suspiciousness_distributions(reflect_on_decision_df, monitors = reflect_on_decision_df.monitor_agent.unique(), metrics = ['response_1_reflection', 'response_2_reflection', 'response_3_reflection', 'average_suspiciousness_reflection', 'max_suspiciousness_reflection'])

In [ ]:
reflect_on_decision_df['1_minus_2'] = reflect_on_decision_df['response_1_reflection'] - reflect_on_decision_df['response_2_reflection']
reflect_on_decision_df['1_minus_3'] = reflect_on_decision_df['response_1_reflection'] - reflect_on_decision_df['response_3_reflection']
reflect_on_decision_df['2_minus_3'] = reflect_on_decision_df['response_2_reflection'] - reflect_on_decision_df['response_3_reflection']
plot_suspiciousness_distributions(reflect_on_decision_df, monitors = reflect_on_decision_df.monitor_agent.unique(), metrics = ['1_minus_2', '1_minus_3', '2_minus_3'])

In [ ]:
merged_df